# 43 · Phim chụp **trước** có mang thông tin không — probe trên embedding đã cache

`42` C1 đo được **71.3%** ảnh có phim trước, đồng đều qua 3 split, `k` hiệu dụng `29`. Nhưng **có
mặt ≠ giúp**. Mô hình longitudinal thật tốn `2×` forward ⇒ ~14 giờ GPU. Notebook này trả lời trước
bằng embedding **đã nằm sẵn trên Drive** cho cả 377,110 ảnh.

## Năm arm, và arm thứ năm là arm quan trọng nhất

```
A  [e_cur]                                768    moc
B  [e_cur ; e_prior]                     1536
C  [e_cur ; e_prior ; e_cur - e_prior]   2304    hieu tuong minh
D  [e_cur ; e_cur - e_prior]             1536    CHI hieu, bo anh truoc
Z  [e_cur ; e_prior CUA BENH NHAN KHAC]  1536    DOI CHUNG KHOP CHIEU
```

**`Z` quyết định mọi thứ.** `B` hơn `A` có thể chỉ vì **nhiều chiều đầu vào hơn** — nhiều tham số,
nhiều khả năng khớp. `Z` có **đúng số chiều** như `B` nhưng phim trước thuộc **bệnh nhân khác**.

```
B - Z  >  0   ->  loi ich den tu DUNG benh nhan do. That.
B - Z ~= 0    ->  loi ich chi la DUNG LUONG. `B - A` la ao anh.
```

Đây đúng khuôn `H4z`/`H6z` của `06b` và `gate_zero` của `37` — đối chứng khớp tham số, không phải
so với arm khác.

## Đọc kết quả theo **một chiều**

`mean(patch)` xoá chi tiết không gian — `37` đo ML-Decoder hơn phép trung bình `+0.0387`. Nên:

```
duong  ->  DI TIEP, dung 14 gio GPU cho mo hinh longitudinal that
null   ->  CHUA dong duoc gi. Chi noi "phep gop trung binh khong thay",
           khong noi "phim truoc vo dung".
```

## Và số nhãn cải thiện quan trọng hơn độ lớn

Theo số học đã lập: đòn bẩy chạm `k` nhãn cần `0.0007 × 40 / k` mỗi nhãn. Cả kỳ vọng của hướng
longitudinal nằm ở chỗ `k ≈ 40` (câu *"cái này có mới không"* áp cho mọi nhãn). `C4` đếm **bao nhiêu
trong 40 nhãn thật sự nhích**, và đó mới là thứ quyết định có đáng 14 giờ hay không.

## Không đụng gì tới `38c`

```
DOC   embeddings/*.npy · embeddings/emb_index.parquet · cxrlt_base.parquet
      cxrlt_index.parquet · cxrlt_inventory.json          (deu chi doc)
GHI   stage4_nb43/   <- thu muc MOI, chua ton tai
```

Chạy ở **runtime riêng**. Nếu Drive là thư mục **được chia sẻ**, `C0` tự dò đường dẫn và **dừng
ngay** nếu tài khoản không có quyền ghi.

## C0 · Config — dò đường dẫn Drive chia sẻ, và kiểm quyền GHI trước khi làm gì

In [2]:
import os, json, math, time, re
import numpy as np, pandas as pd
import torch, torch.nn as nn

try:
    from google.colab import drive; drive.mount('/content/drive'); _COLAB = True
except Exception:
    _COLAB = False

# Drive CHIA SE khong nam o MyDrive. Do vai goc thay vi doan — van de duong dan da lap lai nhieu lan.
CANDS = ['/content/drive/MyDrive/MIMIC_CXR/planB_out',
         '/content/drive/Shareddrives/MIMIC_CXR/planB_out',
         '/content/drive/MyDrive/planB_out',
         './planB_out']
DATA_DIR = next((p for p in CANDS if os.path.isdir(os.path.join(p, 'cxrlt_2024'))), None)
assert DATA_DIR, ('khong thay `cxrlt_2024` o goc nao trong:\n  ' + '\n  '.join(CANDS) +
                  '\nNeu Drive duoc CHIA SE: mo Drive > Shared with me > chuot phai thu muc > '
                  '"Add shortcut to Drive" roi chay lai cell nay.')
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES43   = os.path.join(OUT_DIR, 'stage4_nb43')            # THU MUC MOI, khong dung cua 38c
print(f'DATA_DIR = {DATA_DIR}')

# --- kiem QUYEN GHI ngay bay gio, khong doi den cuoi: Drive chia se co the chi la `viewer` ---
def ensure_res():
    '''Goi TRUOC MOI lan ghi, khong chi mot lan o C0.

       Drive FUSE hay danh roi thu muc RONG — Drive khong thuc su co khai niem thu muc rong nhu
       he tep thuong, nen mot `makedirs` roi khong ghi gi vao se bien mat truoc khi cell sau chay.
       Da dinh dung loi nay: C0 tao thu muc, ghi file thu roi XOA di, den C3 thi
       `Cannot save file into a non-existent directory`.'''
    os.makedirs(RES43, exist_ok=True)
    return RES43

try:
    ensure_res()
    # GHI mot file THAT va GIU lai — thu muc khong bao gio rong nen FUSE khong danh roi
    json.dump({'notebook': '43_prior_probe', 'arm': 'nosq518/patch'},
              open(os.path.join(RES43, '_run_info.json'), 'w'))
    print(f'quyen ghi OK -> {RES43}')
except Exception as _e:
    raise RuntimeError(f'KHONG ghi duoc vao {RES43} ({_e}). Neu Drive duoc chia se, tai khoan nay '
                       'can quyen Editor. Doi RES43 sang /content de chay tam.')

ARM, KIND = 'nosq518', 'patch'     # moc manh nhat cua 32/35 (0.2028)
SEED, SEEDS = 42, (42, 43, 44)
EPOCHS, LR, WD = 60, 3e-3, 1e-4    # asl40 @E=60 — cau hinh 35 da chot
LOSS = 'asl40'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'thiet bi: {dev} (khong bat buoc GPU — head tuyen tinh tren embedding dong cung)')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL            # CUNG cach cat nhu 35/37/38
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
N_POS_TF = YM[TF_].sum(0)
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,}')

Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/MIMIC_CXR/planB_out
quyen ghi OK -> /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb43
thiet bi: cuda (khong bat buoc GPU — head tuyen tinh tren embedding dong cung)
377,110 anh | khop 219,785 | chon 39,086 | val 39,293


## C1 · Ghép phim trước — ở mức **study**, biểu diễn = trung bình ảnh của study đó

Phim trước = study **gần nhất** của cùng bệnh nhân có `index_time` **trước** study hiện tại, và phải
có ảnh đọc được. `e_prior` = trung bình embedding các ảnh của study đó.

Trung bình chứ không chọn frontal: ít tuỳ tiện hơn, và `42` cho thấy trung vị chỉ 2 phim trước nên
phần lớn study trước chỉ có 1–2 ảnh.

In [3]:
BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
BASE['index_time'] = pd.to_datetime(BASE.index_time)
_have = set(ORDER.study_id.unique())
B = BASE[BASE.study_id.isin(_have)].sort_values(['subject_id', 'index_time']).reset_index(drop=True)
B['prev_study'] = B.groupby('subject_id', sort=False).study_id.shift(1)
PREV = dict(zip(B.study_id.to_numpy(), B.prev_study.to_numpy()))    # study -> study truoc (hoac NaN)

# hang ORDER cua tung study, de lay TRUNG BINH embedding
_rows = {}
for i, sid in enumerate(ORDER.study_id.to_numpy()): _rows.setdefault(sid, []).append(i)

prev_of = np.array([PREV.get(s, np.nan) for s in ORDER.study_id.to_numpy()], dtype='float64')
HAS = ~np.isnan(prev_of)
print(f'anh co phim truoc: {HAS.sum():,} / {len(ORDER):,} ({HAS.mean():.1%})')
assert HAS.mean() > 0.5, 'ty le qua thap so voi 71.3% cua nb42 -> kiem lai ghep study'
for nm, m in (('khop', TF_), ('chon', SL), ('val', VA)):
    print(f'  {nm:>4}: {(HAS & m).sum():,} / {m.sum():,} ({(HAS & m).mean():.1%})')

E = np.array(np.load(os.path.join(EMB_DIR, f'emb_{ARM}_{KIND}.npy'), mmap_mode='r'))  # nap HAN
assert E.shape[0] == len(ORDER), f'embedding {E.shape} khong khop {len(ORDER)} hang ORDER'
print(f'embedding {E.shape} ({E.nbytes/1e9:.2f} GB) tu emb_{ARM}_{KIND}.npy')

E_PREV = np.zeros_like(E)
for i in np.where(HAS)[0]:
    E_PREV[i] = E[_rows[int(prev_of[i])]].mean(0)
print('da dung e_prior')

# --- chi giu anh CO phim truoc: so sanh sach, khong lan voi cach dien "khong co tien su" ---
KEEP = HAS
TFk, SLk, VAk = TF_ & KEEP, SL & KEEP, VA & KEEP
print(f'\ntap dung cho probe (CHI anh co phim truoc): khop {TFk.sum():,} | chon {SLk.sum():,} '
      f'| val {VAk.sum():,}')

anh co phim truoc: 253,283 / 377,110 (67.2%)
  khop: 146,515 / 219,785 (38.9%)
  chon: 26,179 / 39,086 (6.9%)
   val: 26,397 / 39,293 (7.0%)
embedding (377110, 768) (1.16 GB) tu emb_nosq518_patch.npy
da dung e_prior

tap dung cho probe (CHI anh co phim truoc): khop 146,515 | chon 26,179 | val 26,397


## C2 · Đầu đọc + chấm điểm — `asl40`, **epoch cuối**, không early-stop

In [4]:
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37. gn=4, gp=0.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        p = torch.sigmoid(x); pm = (p - s.clip).clamp(min=0)
        L = (y * torch.log(p.clamp(min=s.eps)) * (1 - p) ** s.gp
             + (1 - y) * torch.log((1 - pm).clamp(min=s.eps)) * pm ** s.gn)
        return -L.mean()

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o

def fit(X, *, seed=SEED, epochs=EPOCHS):
    '''EPOCH CUOI, cosine trai tren `epochs`. KHONG early-stop (giao thuc nb35 §0.1).
       Chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val.'''
    torch.manual_seed(seed)
    mu, sd = X[TFk].mean(0, keepdims=True), X[TFk].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((X[m] - mu) / sd).astype('float32')).to(dev)
    xt, xs, xv = z(TFk), z(SLk), z(VAk)
    yt = torch.from_numpy(YM[TFk]).to(dev)
    m = nn.Linear(xt.shape[1], C).to(dev)
    crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    bs = int(min(4096, max(256, len(xt) // 64)))          # san 64 buoc/epoch (bay da dinh o nb35)
    n = len(xt)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            k = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            crit(m(xt[k]), yt[k]).backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad():
        pv = torch.sigmoid(m(xv)).cpu().numpy(); ps = torch.sigmoid(m(xs)).cpu().numpy()
    return pv, ps

tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
print(f'san sang | {LOSS} E={EPOCHS} epoch cuoi | seeds {SEEDS}')

san sang | asl40 E=60 epoch cuoi | seeds (42, 43, 44)


## C3 · Năm arm × ba seed

`Z` dùng hoán vị **theo bệnh nhân** (không phải theo hàng): mỗi ảnh nhận `e_prior` của một bệnh nhân
khác. Cùng số chiều với `B`, cùng phân phối giá trị — chỉ khác **nó không thuộc về bệnh nhân này**.

In [5]:
_rs = np.random.default_rng(0)
_idx = np.where(KEEP)[0]
_perm = np.zeros(len(ORDER), 'int64'); _perm[_idx] = _rs.permutation(_idx)
E_SHUF = np.zeros_like(E_PREV); E_SHUF[_idx] = E_PREV[_perm[_idx]]
_same = (SUB[_idx] == SUB[_perm[_idx]]).mean()
print(f'hoan vi Z: {_same:.2%} truong hop tinh co van cung benh nhan (nho la dung)')

def build(tag):
    if tag == 'A': return E
    if tag == 'B': return np.concatenate([E, E_PREV], 1)
    if tag == 'C': return np.concatenate([E, E_PREV, E - E_PREV], 1)
    if tag == 'D': return np.concatenate([E, E - E_PREV], 1)
    if tag == 'Z': return np.concatenate([E, E_SHUF], 1)
ARMS = {'A': '[e_cur]', 'B': '[e_cur;e_prior]', 'C': '[e_cur;e_prior;hieu]',
        'D': '[e_cur;hieu]', 'Z': '[e_cur;e_prior BN KHAC]  <- DOI CHUNG'}

P_RES = os.path.join(RES43, 'probe.csv')
R = pd.read_csv(P_RES) if os.path.exists(P_RES) else pd.DataFrame()
PRED = {}
for tag, desc in ARMS.items():
    X = build(tag)
    for sd_ in SEEDS:
        if len(R) and ((R.arm == tag) & (R.seed == sd_)).any():
            print(f'{tag} s{sd_}: da co, bo qua'); continue
        t0 = time.time(); pv, ps = fit(X, seed=sd_)
        a = ap_cols(YM[VAk], pv)
        row = dict(arm=tag, mo_ta=desc, seed=sd_, D=X.shape[1],
                   mAP_anh=float(np.nanmean(a)), **tiers(a),
                   sel=float(np.nanmean(ap_cols(YM[SLk], ps))))
        R = pd.concat([R, pd.DataFrame([row])], ignore_index=True)
        ensure_res()                       # Drive FUSE co the da danh roi thu muc giua hai cell
        R.to_csv(P_RES, index=False)
        np.save(os.path.join(RES43, f'p_{tag}_s{sd_}.npy'), pv)
        print(f'{tag} s{sd_}  D={X.shape[1]:>4}  mAP {row["mAP_anh"]:.4f}  tail {row["tail"]:.4f}  '
              f'lat chon {row["sel"]:.4f}  ({time.time()-t0:.0f}s)')
    del X
print('\nxong')

hoan vi Z: 0.01% truong hop tinh co van cung benh nhan (nho la dung)
A s42  D= 768  mAP 0.2057  tail 0.0516  lat chon 0.1972  (9s)
A s43  D= 768  mAP 0.2061  tail 0.0530  lat chon 0.1971  (3s)
A s44  D= 768  mAP 0.2069  tail 0.0552  lat chon 0.1973  (3s)
B s42  D=1536  mAP 0.2080  tail 0.0520  lat chon 0.1994  (3s)
B s43  D=1536  mAP 0.2089  tail 0.0542  lat chon 0.1987  (3s)
B s44  D=1536  mAP 0.2097  tail 0.0575  lat chon 0.1999  (3s)
C s42  D=2304  mAP 0.2070  tail 0.0523  lat chon 0.1966  (3s)
C s43  D=2304  mAP 0.2072  tail 0.0524  lat chon 0.1970  (3s)
C s44  D=2304  mAP 0.2074  tail 0.0542  lat chon 0.1970  (3s)
D s42  D=1536  mAP 0.2092  tail 0.0532  lat chon 0.1981  (3s)
D s43  D=1536  mAP 0.2093  tail 0.0538  lat chon 0.1987  (3s)
D s44  D=1536  mAP 0.2095  tail 0.0550  lat chon 0.1998  (3s)
Z s42  D=1536  mAP 0.1973  tail 0.0418  lat chon 0.1885  (3s)
Z s43  D=1536  mAP 0.1970  tail 0.0409  lat chon 0.1881  (3s)
Z s44  D=1536  mAP 0.1976  tail 0.0422  lat chon 0.1887  (3s)



## C4 · Đọc kết quả — `B − Z` là dòng quyết định, và **đếm số nhãn**

In [6]:
assert os.path.exists(P_RES), f'khong thay {P_RES} -> chay lai C3'
R = pd.read_csv(P_RES)
piv = R.pivot_table(index='arm', values=['mAP_anh', 'head', 'medium', 'tail', 'sel'], aggfunc='mean')
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max() - s.min())
piv['D'] = R.groupby('arm').D.first()
print('=== TUYET DOI (TB qua seed) ===')
print(piv.reindex(list(ARMS)).to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(piv.bien_do.max())
print(f'\nbien do seed lon nhat qua cac arm: {NOISE:.4f}  <- nguong nhieu CUA CHINH phep do nay')

g = lambda t, s: float(R[(R.arm == t) & (R.seed == s)].mAP_anh.iloc[0])
print('\n=== HIEU GHEP CAP ===')
for a, b, nghia in (('B', 'A', 'them phim truoc (co the chi la them CHIEU)'),
                    ('C', 'A', 'them phim truoc + hieu'),
                    ('D', 'A', 'CHI hieu'),
                    ('Z', 'A', 'DOI CHUNG: them chieu, phim truoc SAI benh nhan'),
                    ('B', 'Z', '<<< DONG QUYET DINH: loi ich co den tu DUNG benh nhan khong')):
    d = [g(a, s) - g(b, s) for s in SEEDS]
    ok = all(x > 0 for x in d) or all(x < 0 for x in d)
    print(f'  {a} - {b}  {np.mean(d):+.4f}  (bien do {max(d)-min(d):.4f}, '
          f'{"cung dau" if ok else "DOI DAU"})   {nghia}')

print('\n=== DEM SO NHAN — day moi quyet dinh co dang 14 gio GPU khong ===')
_pa = np.mean([np.load(os.path.join(RES43, f'p_A_s{s}.npy')) for s in SEEDS], 0)
for t in ('B', 'C', 'D', 'Z'):
    _pt = np.mean([np.load(os.path.join(RES43, f'p_{t}_s{s}.npy')) for s in SEEDS], 0)
    dl = ap_cols(YM[VAk], _pt) - ap_cols(YM[VAk], _pa)
    k = int((dl > NOISE).sum())
    print(f'  {t} - A: {k:>2}/40 nhan vuot nguong nhieu | trung vi {np.nanmedian(dl):+.5f} '
          f'| macro can neu k={k}: {0.0007*40/max(k,1):.4f}')
    if t == 'B':
        _top = np.argsort(-dl)[:6]
        print('     nhan len nhieu nhat: ' + ' · '.join(f'{LAB40[j]} {dl[j]:+.4f}' for j in _top))
print('\n  DOC MOT CHIEU: duong -> di tiep, dung 14 gio GPU cho mo hinh longitudinal that.')
print('  null -> CHUA dong duoc gi: mean(patch) xoa chi tiet khong gian (37: ML-Decoder hon')
print('          phep trung binh +0.0387), nen null o day chi noi "phep gop trung binh khong thay".')

=== TUYET DOI (TB qua seed) ===
      head  mAP_anh  medium    sel   tail  bien_do     D
arm                                                     
A   0.4265   0.2062  0.1324 0.1972 0.0533   0.0012   768
B   0.4313   0.2089  0.1342 0.1993 0.0546   0.0017  1536
C   0.4303   0.2072  0.1320 0.1969 0.0530   0.0004  2304
D   0.4316   0.2093  0.1356 0.1989 0.0540   0.0003  1536
Z   0.4221   0.1973  0.1217 0.1884 0.0416   0.0006  1536

bien do seed lon nhat qua cac arm: 0.0017  <- nguong nhieu CUA CHINH phep do nay

=== HIEU GHEP CAP ===
  B - A  +0.0026  (bien do 0.0005, cung dau)   them phim truoc (co the chi la them CHIEU)
  C - A  +0.0010  (bien do 0.0009, cung dau)   them phim truoc + hieu
  D - A  +0.0031  (bien do 0.0009, cung dau)   CHI hieu
  Z - A  -0.0089  (bien do 0.0009, cung dau)   DOI CHUNG: them chieu, phim truoc SAI benh nhan
  B - Z  +0.0116  (bien do 0.0014, cung dau)   <<< DONG QUYET DINH: loi ich co den tu DUNG benh nhan khong

=== DEM SO NHAN — day moi quyet dinh co dang 

## C5 · Ghi `REVIEW_PRIOR_PROBE.md`

In [7]:
L = []; w = L.append
w('# REVIEW · phim chụp trước có mang thông tin không (`nb43`)'); w('')
w(f'`{ARM}/{KIND}` · `{LOSS} E={EPOCHS}` epoch cuối · seeds {list(SEEDS)} · '
  f'chỉ ảnh **có** phim trước (val {int(VAk.sum()):,} ảnh).'); w('')
w('| arm | biểu diễn | D | mAP ảnh | biên độ | tail |')
w('|---|---|---|---|---|---|')
for t in ARMS:
    r = R[R.arm == t]
    w(f'| `{t}` | `{ARMS[t]}` | {int(r.D.iloc[0])} | {r.mAP_anh.mean():.4f} | '
      f'{r.mAP_anh.max()-r.mAP_anh.min():.4f} | {r["tail"].mean():.4f} |')
w('')
_bz = np.mean([g('B', s) - g('Z', s) for s in SEEDS])
_ba = np.mean([g('B', s) - g('A', s) for s in SEEDS])
w(f'```'); w(f'B - A  {_ba:+.4f}   them phim truoc (co the chi la them CHIEU)')
w(f'B - Z  {_bz:+.4f}   <<< loi ich co den tu DUNG benh nhan khong'); w('```'); w('')
w(f'**{"CÓ TÍN HIỆU — đi tiếp" if _bz > NOISE else "CHƯA vượt nhiễu"}** '
  f'(biên độ seed của chính phép đo này: `{NOISE:.4f}`)'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **`mean(patch)` xoá chi tiết không gian.** `37` đo ML-Decoder hơn phép trung bình `+0.0387`. '
  'Null ở đây chỉ nói *"phép gộp trung bình không thấy"*, **không** nói *"phim trước vô dụng"*.')
w('- **Cơ chế.** `42` đo `p25 = 0.9 ngày` — một phần tư phim trước là phim nối tiếp trong cùng đợt '
  'nằm viện, nơi nhãn gần như bất biến. Lợi ích (nếu có) phần lớn là **tự tương quan nhãn trong một '
  'đợt điều trị**, không phải phát hiện thay đổi. Phải gọi đúng tên khi viết bài.')
w('- **28.7% ảnh không có phim trước** bị loại khỏi phép đo này. Mô hình thật phải xử lý chúng bằng '
  'một token "không có tiền sử" — chưa đo ở đây.')
w('- Chỉ một backbone (`RAD-DINO` đông cứng) và một phép gộp (`mean`).')
TXT = '\n'.join(L)
ensure_res()
open(os.path.join(RES43, 'REVIEW_PRIOR_PROBE.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REVIEW · phim chụp trước có mang thông tin không (`nb43`)

`nosq518/patch` · `asl40 E=60` epoch cuối · seeds [42, 43, 44] · chỉ ảnh **có** phim trước (val 26,397 ảnh).

| arm | biểu diễn | D | mAP ảnh | biên độ | tail |
|---|---|---|---|---|---|
| `A` | `[e_cur]` | 768 | 0.2062 | 0.0012 | 0.0533 |
| `B` | `[e_cur;e_prior]` | 1536 | 0.2089 | 0.0017 | 0.0546 |
| `C` | `[e_cur;e_prior;hieu]` | 2304 | 0.2072 | 0.0004 | 0.0530 |
| `D` | `[e_cur;hieu]` | 1536 | 0.2093 | 0.0003 | 0.0540 |
| `Z` | `[e_cur;e_prior BN KHAC]  <- DOI CHUNG` | 1536 | 0.1973 | 0.0006 | 0.0416 |

```
B - A  +0.0026   them phim truoc (co the chi la them CHIEU)
B - Z  +0.0116   <<< loi ich co den tu DUNG benh nhan khong
```

**CÓ TÍN HIỆU — đi tiếp** (biên độ seed của chính phép đo này: `0.0017`)

## Những gì kết quả này KHÔNG trả lời

- **`mean(patch)` xoá chi tiết không gian.** `37` đo ML-Decoder hơn phép trung bình `+0.0387`. Null ở đây chỉ nói *"phép gộp trung bình không thấy"*, **không** nói *"phim trước vô dụng